## Лабораторный практикум по курсу «Разговорный ИИ и большие языковые модели», Университет ИТМО, 2026  
**Лабораторная работа №3**

### Задание

1. Выбрать QA датасет (https://huggingface.co/datasets). 
2. На выбранном датасете обучить модель https://huggingface.co/intfloat/multilingual-e5-small с помощью этого кода.
3. Переписать код под актуальную версию sentence-transformers.
4. Сравнить результаты

In [1]:
# pip install transformers==4.45.2 
# pip install sentence-transformers==3.1.1

In [2]:
import pickle
from random import shuffle

from datasets import load_dataset
from sentence_transformers import InputExample

In [3]:
import torch

device = 'mps' if torch.backends.mps.is_built() else 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

cuda


In [4]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.modules import Transformer, Pooling

def raw_bi_encoder():
    word_embedding_model = Transformer('intfloat/multilingual-e5-small', max_seq_length=256)
    pooling_model = Pooling(word_embedding_model.get_embedding_dimension())

    bi_encoder = SentenceTransformer(
        modules=[word_embedding_model, pooling_model],
        device=device,
    )

    return bi_encoder

In [6]:
bi_encoder = raw_bi_encoder()

print(bi_encoder)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 469.52it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)


## Служебные функции для работы с данными

In [8]:
def shrink_repeated_samples(
        queries,
        docs,
        labels,
):
    docs_was = set()
    qs = []
    ds = []
    ls = []
    for i in range(len(queries)):
        q, d, l = queries[i], docs[i], labels[i]
        if d in docs_was:
            continue
        qs.append(q)
        ds.append(d)
        ls.append(l)
        docs_was.add(d)

    return qs, ds, ls


def trainval_shuffled_data(from_file=True):
    datafile = 'trainval.pkl'

    if from_file:
        with open(datafile, 'rb') as f:
            data = pickle.load(f)
        return data

    good, bad = load_trainval_samples()
    data = good + bad
    shuffle(data)

    with open(datafile, 'wb') as f:
        pickle.dump(data, f)

    return data


def test_shuffled_data(from_file=True):
    datafile = 'test.pkl'

    if from_file:
        with open(datafile, 'rb') as f:
            data = pickle.load(f)
        return data

    good, bad = load_test_samples()
    data = good + bad

    with open(datafile, 'wb') as f:
        pickle.dump(data, f)

    return data

def trainval_examples():
    data = trainval_shuffled_data(from_file=True)

    trainval = []
    for question, document, cos in data:
        trainval.append(
            InputExample(texts=[f"query: {question}", f"passage: {document}"], label=cos)
        )

    return trainval

def load_test_samples():
    _, _, test = _load_sberquad_dataset()

    questions = test['question'].tolist()
    contexts = test['context'].tolist()

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def load_trainval_samples():
    train, val, _ = _load_sberquad_dataset()

    questions = train['question'].tolist() + val['question'].tolist()
    contexts = train['context'].tolist() + val['context'].tolist()

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def _inflate_with_negative_samples(
        questions,
        contexts,
        delta=30,  # in sberquad dataset same contexts are close
):
    good_q_c_cos = list(zip(questions, contexts, [1.0] * len(questions)))

    bad_q_c_cos = []
    n = len(good_q_c_cos)
    for i in range(n):
        cur_q, cur_c, _ = good_q_c_cos[i]
        next_q, next_c, _ = good_q_c_cos[(i + delta) % n]
        if next_c != cur_c:
            bad_q_c_cos.append((cur_q, next_c, 0.0))

    return good_q_c_cos, bad_q_c_cos


def _load_raw_sberquad_dataset(from_file=True):
    sberquad = 'sberquad.pkl'
    if from_file:
        with open(sberquad, 'rb') as f:
            df = pickle.load(f)
    else:
        df = load_dataset('sberquad')
        with open(sberquad, 'wb') as f:
            pickle.dump(df, f)

    return df


def _load_sberquad_dataset():
    df = _load_raw_sberquad_dataset()
    train = df['train'].to_pandas()
    validation = df['validation'].to_pandas()
    test = df['test'].to_pandas()

    return train, validation, test

# Метрики близости без дообучения на trainval

In [10]:
trainval = trainval_shuffled_data(from_file=True)

n_samples = 1000
# questions, document_contexts, coss = zip(*trainval[:n_samples])
questions, document_contexts, coss = zip(*trainval)

In [11]:
from sentence_transformers.sentence_transformer import evaluation
evaluator = evaluation.EmbeddingSimilarityEvaluator(questions, document_contexts, coss)

In [83]:
bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.9146496736745333, 'spearman_cosine': 0.8633254532294019}

# Метрики близости без дообучения на test

In [12]:
n_test_samples = 1000
test = test_shuffled_data(from_file=True)
test_questions, test_document_contexts, test_coss = zip(*test[:n_test_samples])

In [13]:
evaluator = evaluation.EmbeddingSimilarityEvaluator(test_questions, test_document_contexts, test_coss)

In [86]:
bi_encoder = raw_bi_encoder()

Loading weights: 100%|██████████| 199/199 [00:02<00:00, 81.33it/s]


In [87]:
bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.9195757348519364, 'spearman_cosine': 0.8630726356761861}

# Дообучение

In [88]:
# trainval_examples = trainval_examples()[:n_samples]
trainval_examples = trainval_examples()
print(trainval_examples[0].__dict__)

{'guid': '', 'texts': ['query: что оказала Заметное, хотя и менее значительное влияние на воззрения Локка', 'passage: Заметное, хотя и менее значительное влияние на воззрения Локка оказала психология Гоббса, у которого заимствован, например, порядок изложения Опыта . Описывая процессы сравнения, Локк следует за Гоббсом; вместе с ним он утверждает, что отношения не принадлежат вещам, а составляют результат сравнения, что отношений бесчисленное множество, что более важные отношения суть тождество и различие, равенство и неравенство, сходство и несходство, смежность по пространству и времени, причина и действие. В трактате о языке, то есть в третьей книге Опыта , Локк развивает мысли Гоббса. В учении о воле Локк находится в сильнейшей зависимости от Гоббса; вместе с последним он учит, что стремление к удовольствию есть единственное проходящее через всю нашу психическую жизнь и что понятие о добре и зле у различных людей совершенно различно. В учении о свободе воли Локк вместе с Гоббсом ут

In [ ]:
from torch.utils.data import DataLoader

trainval_dataloader_set = DataLoader(trainval_examples, shuffle=True, batch_size=32)

In [90]:
from sentence_transformers.sentence_transformer.losses import CosineSimilarityLoss

trainval_loss_set = CosineSimilarityLoss(bi_encoder)

In [91]:
trainval_loss_set

CosineSimilarityLoss(
  (model): SentenceTransformer(
    (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
    (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  )
  (loss_fct): MSELoss()
  (cos_score_transformation): Identity()
)

In [92]:
(query_batch, context_batch), labels = next(iter(trainval_dataloader_set))
query_batch['input_ids'].shape, context_batch['input_ids'].shape, labels.shape

(torch.Size([32, 79]), torch.Size([32, 256]), torch.Size([32]))

In [ ]:
bi_encoder.fit(
    train_objectives=[(trainval_dataloader_set, trainval_loss_set)],
    output_path='qa/results_new',
    epochs=5,
    evaluator=evaluator,
    evaluation_steps=1000,
    show_progress_bar=True,
)

Step,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine,Pearson Manhattan,Spearman Manhattan,Pearson Euclidean,Spearman Euclidean,Pearson Dot,Spearman Dot,Pearson Max,Spearman Max
1000,0.077100,No log,0.899779,0.851710,0.885724,0.846618,0.886916,0.847186,0.885175,0.848204,0.899779,0.851710
2000,0.062900,No log,0.905001,0.852334,0.893241,0.849528,0.894099,0.849528,0.890728,0.848980,0.905001,0.852334
3000,0.059000,No log,0.911038,0.853380,0.897311,0.850678,0.898344,0.850768,0.896417,0.851676,0.911038,0.853380
3148,0.059000,No log,0.910181,0.853269,0.896725,0.850615,0.897645,0.850574,0.894485,0.851100,0.910181,0.853269
4000,0.053800,No log,0.907302,0.851253,0.892784,0.848142,0.894066,0.848419,0.892733,0.849514,0.907302,0.851253
5000,0.052100,No log,0.911089,0.852264,0.897533,0.850054,0.898884,0.850484,0.892727,0.850089,0.911089,0.852264
6000,0.050700,No log,0.913516,0.854294,0.897430,0.851918,0.898559,0.852292,0.898343,0.852549,0.913516,0.854294
6296,0.050700,No log,0.913536,0.854689,0.900813,0.853255,0.901535,0.853352,0.897221,0.852258,0.913536,0.854689
7000,0.046000,No log,0.911721,0.853117,0.896569,0.850740,0.897161,0.850609,0.896096,0.851170,0.911721,0.853117
8000,0.045300,No log,0.907919,0.851066,0.895803,0.849431,0.896325,0.849528,0.892527,0.848565,0.907919,0.851066


# Загрузка чекпоинта

In [22]:
from sentence_transformers import SentenceTransformer

finetuned_bi_encoder = SentenceTransformer('qa/results_new')

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 141.83it/s]


In [50]:
finetuned_bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.9146832987963025, 'spearman_cosine': 0.8549526312689782}

# Расчёт метрик

In [14]:
data = test_shuffled_data(from_file=True)

In [15]:
n_test_samples = 1000
queries, docs, labels = zip(*data[:n_test_samples])

In [16]:
print('docs count before shrinking: ', len(docs))

docs count before shrinking:  1000


In [17]:
docs = list(set(docs))

In [18]:
print('docs count without duplicates: ', len(docs))

docs count without duplicates:  891


In [23]:
doc_embs = finetuned_bi_encoder.encode(
    [f"passage: {d}" for d in docs],
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 28/28 [00:02<00:00, 13.87it/s]


In [24]:
query_embs = finetuned_bi_encoder.encode(
    [f"query: {q}" for q in queries],
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 32/32 [00:00<00:00, 74.62it/s]


In [20]:
from sentence_transformers import util

In [25]:
cos_sim_score = util.semantic_search(query_embs, doc_embs, top_k=3)

In [60]:
for_first_query = cos_sim_score[0]
print(for_first_query)

[{'corpus_id': 342, 'score': 0.8746881484985352}, {'corpus_id': 836, 'score': 0.8240224123001099}, {'corpus_id': 119, 'score': 0.7985023260116577}]


In [61]:
print('Question: ', queries[0])
print()
for no, ir in enumerate(cos_sim_score[0]):
    corpus_id = ir["corpus_id"]
    print('corpus id: ', corpus_id)
    print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
    print()

Question:  Какое содержание селена в морской воде?

corpus id:  342
Document 1: Cosine Similarity is 0.875:

Хотя химический состав озёр остаётся относительно длительное время постоянным, в отличие от реки заполняющее его вещество обновляется значительно реже, а имеющиеся в нём течения не являются преобладающим фактором, определяющим его режим. Озёра регулируют сток рек, задерживая в своих котловинах полые воды и отдавая их в другие периоды. В водах озёр происходят химические реакции. Одни элементы переходят из воды в донные отложения, другие — наоборот. В ряде озёр, главным образом не имеющих стока, в связи с испарением воды повышается концентрация солей. Результатом являются существенные изменения минерализации и солевого состава озёр. Благодаря значительной тепловой инерции водной массы крупные озёра смягчают климат прилегающих районов, уменьшая годовые и сезонные колебания метеорологических элементов.

corpus id:  836
Document 2: Cosine Similarity is 0.824:

Содержание селена в зем

In [62]:
data = test_shuffled_data(from_file=True)
n_test_samples = 1000
queries, docs, labels = zip(*data[:n_test_samples])

queries, docs, labels = shrink_repeated_samples(
    queries=queries,
    docs=docs,
    labels=labels,
)
print(len(docs))

891


# Получение предсказаний модели и меток

In [63]:
doc_embs = finetuned_bi_encoder.encode(
    [f"passage: {d}" for d in docs],
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 28/28 [00:01<00:00, 14.33it/s]


In [64]:
query_embs = finetuned_bi_encoder.encode(
    [f"query: {q}" for q in queries],
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 28/28 [00:00<00:00, 83.40it/s]


In [65]:
cos_scores = util.semantic_search(query_embs, doc_embs, top_k=100)

In [66]:
print(len(cos_scores))

891


In [67]:
print(cos_scores[0][:5])

[{'corpus_id': 376, 'score': 0.8746881484985352}, {'corpus_id': 0, 'score': 0.8240224123001099}, {'corpus_id': 671, 'score': 0.7985023260116577}, {'corpus_id': 271, 'score': 0.7807059288024902}, {'corpus_id': 183, 'score': 0.7776356935501099}]


In [68]:
top_k = 5

In [69]:
print('Question: ', queries[0])
print()
for no, ir in enumerate(cos_scores[0][:top_k]):
    corpus_id = ir["corpus_id"]
    print('corpus id: ', corpus_id)
    print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
    print()

Question:  Какое содержание селена в морской воде?

corpus id:  376
Document 1: Cosine Similarity is 0.875:

Хотя химический состав озёр остаётся относительно длительное время постоянным, в отличие от реки заполняющее его вещество обновляется значительно реже, а имеющиеся в нём течения не являются преобладающим фактором, определяющим его режим. Озёра регулируют сток рек, задерживая в своих котловинах полые воды и отдавая их в другие периоды. В водах озёр происходят химические реакции. Одни элементы переходят из воды в донные отложения, другие — наоборот. В ряде озёр, главным образом не имеющих стока, в связи с испарением воды повышается концентрация солей. Результатом являются существенные изменения минерализации и солевого состава озёр. Благодаря значительной тепловой инерции водной массы крупные озёра смягчают климат прилегающих районов, уменьшая годовые и сезонные колебания метеорологических элементов.

corpus id:  0
Document 2: Cosine Similarity is 0.824:

Содержание селена в земно

In [70]:
print(docs[582])

К хлоридно-гидрокарбонатно-натриевому типу относится закарпатская вода Драговская с минерализацией 9,6 г/л и краснодарская Горячий ключ с общим содержанием солей в литре 4,5 г солей, но в них хлориды в виде поваренной соли (соответственно 59 и 67 %) превалируют над гидрокарбонатами, которые представлены содой (38 и 32 %). Обе воды безсульфатные. Преобладанием хлоридов над гидрокарбонатами отличается и вода того же типа Челкарская с минерализацией 2,2 г/л. Гидрокарбонаты в виде соды составляют 32, а хлоридов (поваренная соль) — 48 %. Кроме того, в Челкарской есть сульфаты в виде глауберовой соли (20 %).


In [71]:
n = len(queries)
print(n)

891


In [72]:
def build_target_mask_for_i(i, n):
    target_mask = [False] * n
    target_mask[i] = True

    return target_mask

In [73]:
def build_pred_mask_for_i(
        cos_scores,
        i,
        n,
):
    pred_mask = [0.0] * n
    qi_scores = cos_scores[i]

    for docid_score in qi_scores:
        doc_id = docid_score['corpus_id']
        score = docid_score['score']
        pred_mask[int(doc_id)] = score

    return pred_mask

# Расчёт метрик: Recall@5, MRR, MAP, NDCG@10

In [74]:
from statistics import mean
from torchmetrics.functional.retrieval import retrieval_reciprocal_rank, retrieval_average_precision, \
    retrieval_normalized_dcg, retrieval_recall

r5s = []
mrrs = []
rmaps = []
ndcgs = []
for i in range(n):
    pred = build_pred_mask_for_i(cos_scores, i=i, n=n)
    target = build_target_mask_for_i(i=i, n=n)

    pred = torch.tensor(pred)
    target = torch.tensor(target)
    
    r5 = retrieval_recall(preds=pred, target=target, top_k=5).item()
    mrr = retrieval_reciprocal_rank(preds=pred, target=target, top_k=5).item()
    rmap = retrieval_average_precision(preds=pred, target=target, top_k=5).item()
    ndcg = retrieval_normalized_dcg(preds=pred, target=target, top_k=10).item()
    
    r5s.append(r5)
    mrrs.append(mrr)
    rmaps.append(rmap)
    ndcgs.append(ndcg)

print('mean recall5@891: ', mean(r5s))
print('mean mrr: ', mean(mrrs))
print('mean rmAP: ', mean(rmaps))
print('mean ndcg10@891: ', mean(ndcgs))

mean recall5@891:  0.42424242424242425
mean mrr:  0.3684062853405371
mean rmAP:  0.3684062853405371
mean ndcg10@891:  0.38987171797104825


## Изменения при адаптации под актуальную версию sentence-transformers

Код был переписан под `sentence-transformers 6.2.0` и `transformers 5.20.2`.  
Основные изменения:

| Компонент | Было | Стало |
| :--- | :--- | :--- |
| **Версия `sentence-transformers`** | `3.1.1` | `6.2.0` |
| **Версия `transformers`** | `4.45.2` | `5.20.2` |
| **Метод размерности** | `get_word_embedding_dimension()` | `get_embedding_dimension()` |
| **Импорт модулей** | `from sentence_transformers import models`<br>`models.Transformer`, `models.Pooling` | `from sentence_transformers.sentence_transformer.modules import Transformer, Pooling`<br>`Transformer`, `Pooling` |
| **Префиксы E5** | Отсутствовали | `query: ` и `passage: ` добавлены в `InputExample` и `encode` |
| **Импорт evaluator** | `from sentence_transformers import evaluation` | `from sentence_transformers.sentence_transformer import evaluation` |
| **Импорт losses** | `from sentence_transformers import losses` | `from sentence_transformers.sentence_transformer.losses import CosineSimilarityLoss` |

## Сравнение результатов

| Метрика | До обучения | Старые версии библиотек | Адаптированный под новые версии код |
| :--- | :--- | :--- | :--- |
| **Recall@5** | 0.489 | 0.424 | 0.424 |
| **MRR** | 0.469 | 0.371 | 0.368 |
| **rmAP** | 0.469 | 0.371 | 0.368 |
| **nDCG@10** | 0.476 | 0.393 | 0.390 |

**Вывод:**  

1. Baseline (E5-small без дообучения) показывает более высокие метрики, чем после дообучения. Это объясняется:
   - Слабыми негативами (случайный сдвиг на 30 позиций не учитывает малые различия);
   - Малым размером test (891 документ);
   - E5-small уже хорошо обучена, дообучение на маленьком датасете с простыми негативами ухудшает обобщение.
2. Результаты двух вариантов **практически идентичны** (разница ≤ 0.003), адаптация кода под новую версию `sentence-transformers` не сломала обучение и не ухудшила качество. Небольшие расхождения объясняются стохастичностью обучения (не зафиксирован `seed`) и разным порядком документов из-за `list(set(docs))`.

In [ ]:
baseline_bi_encoder = raw_bi_encoder()

doc_embs_baseline = baseline_bi_encoder.encode(
    [f"passage: {d}" for d in docs],
    convert_to_tensor=True,
    show_progress_bar=True,
)
query_embs_baseline = baseline_bi_encoder.encode(
    [f"query: {q}" for q in queries],
    convert_to_tensor=True,
    show_progress_bar=True,
)

cos_scores_baseline = util.semantic_search(query_embs_baseline, doc_embs_baseline, top_k=100)

r5s_b, mrrs_b, rmaps_b, ndcgs_b = [], [], [], []
for i in range(n):
    pred = build_pred_mask_for_i(cos_scores_baseline, i=i, n=n)
    target = build_target_mask_for_i(i=i, n=n)
    pred = torch.tensor(pred)
    target = torch.tensor(target)

    r5s_b.append(retrieval_recall(preds=pred, target=target, top_k=5).item())
    mrrs_b.append(retrieval_reciprocal_rank(preds=pred, target=target, top_k=5).item())
    rmaps_b.append(retrieval_average_precision(preds=pred, target=target, top_k=5).item())
    ndcgs_b.append(retrieval_normalized_dcg(preds=pred, target=target, top_k=10).item())

print('BASELINE (до обучения):')
print('mean recall5: ', mean(r5s_b))
print('mean mrr: ', mean(mrrs_b))
print('mean rmAP: ', mean(rmaps_b))
print('mean ndcg10: ', mean(ndcgs_b))

Batches: 100%|██████████| 28/28 [00:00<00:00, 76.00it/s]


BASELINE (до обучения):
mean recall5:  0.489337822671156
mean mrr:  0.4686120464544906
mean rmAP:  0.4686120464544906
mean ndcg10:  0.47555663984365065
